# Configuration

## Imports

In [ ]:
import numpy as np
from math import sin, cos, tan, asin, atan, radians, degrees
import shapely.geometry as sg
import matplotlib.pyplot as plt
import descartes
from scipy import constants


import platosim.referenceFrames as rf

## Configuration parameters

In [ ]:
focalLength = 247.52     # [mm]
pixelSize = 18     # [µm]
plateScale = 15     # [arcsec]
numGroups = 4
numCorners = 4
ccdCodes = ["1", "2", "3", "4"]
tiltAngles = [9.2, 9.2, 9.2, 9.2]     # [degrees]
azimuthAngles = [45.0, 135.0, 225.0, 315.0]     # [degrees]

In [ ]:
# Size of the FOV

fovDegrees = 18.8908
fovPixels = fovDegrees / plateScale * constants.degree / constants.arcsec
fovMm = focalLength * tan(radians(fovDegrees))

In [ ]:
colors = ['b', 'r', 'g', 'orange']

## Auxiliary functions

In [ ]:
# def rotate_origin_only(x, y, angle):

#     """
#     Rotate point (x, y) around the origin over the given angle.
    
#     :param x: Point's x-coordinate.
    
#     :param y: Point's y-coordinate.
    
#     :param angle: Rotation angle [degrees].  A positive (negative) angle corresponds to
#                   a counterclockwise (clockwise) rotation.
    
#     :return xRot: Point's x-coordinate after rotation.
    
#     :return yRot: Point's y-coordinate after rotation.
#     """
    
#     angleRadians = radians(angle)     # [degrees] -> [radians]
    
#     xRot = cos(angleRadians) * x - sin(angleRadians) * y
#     yRot = sin(angleRadians) * x + cos(angleRadians) * y

#     return xRot, yRot

In [ ]:
def mm2pixels(distanceMm):
    
    """
    Conversion from millimeters to pixels.
    
    :param distanceMm: Distance [mm].
    
    :return distancePixels: Distance [pixels].
    """
    
    distancePixels = degrees(atan(distanceMm / focalLength)) / plateScale * constants.degree / constants.arcsec
    
    return distancePixels

In [ ]:
sign = lambda x: (1, -1)[x < 0]

# Telescope pointing w.r.t. platform pointing

## Sky

In [ ]:
# Arbitrary choice of platform pointing and solar-panel orientation

raPlatform = 10.               # [degrees]
decPlatform = 10.              # [degrees]
solarPanelOrientation = 0.     # [degrees]

raSun, decSun = rf.sunSkyCoordinatesAwayfromPlatformPointing(radians(raPlatform), radians(decPlatform), solarPanelOrientation)

In [ ]:
raTelescope = []
decTelescope = []

for group in range(0, numGroups):
    
    # Telescope pointing (absolute)
    
    ra, dec = rf.platformToTelescopePointingCoordinates(radians(raPlatform), radians(decPlatform), radians(solarPanelOrientation),
                                                        radians(azimuthAngles[group]), radians(tiltAngles[group]))     # [radians]
    
    # Telescope pointing w.r.t. platform pointing
    
    raTelescope.append(degrees(ra) - raPlatform)     # [degrees]
    decTelescope.append(degrees(dec) - decPlatform)  # [degrees]
    
meanDist = (np.mean(np.absolute(raTelescope)) + np.mean(np.absolute(decTelescope))) / 2.0

for group in range(numGroups):
    
    raTelescope[group] = sign(raTelescope[group]) * meanDist
    decTelescope[group] = sign(decTelescope[group]) * meanDist

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111)

ax.plot([raPlatform - raPlatform], [decPlatform - decPlatform], "ro")
ax.plot(raTelescope, decTelescope, "bo")

offset = 1

for group in range(numGroups):
    
    ax.text(raTelescope[group] + offset, decTelescope[group] + offset, "Group " + str(group + 1), fontsize = 20)

ax.text(0 + offset, 0 + offset, "$z_{PLM}$", fontsize = 20)
    
circles = []
for group in range(numGroups):
    circles.append(sg.Point(raTelescope[group], decTelescope[group]).buffer(fovDegrees))

for index in range(numGroups):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))
    
ax.set_title('Telescope pointing w.r.t. platform pointing', fontsize = 24)
ax.set_xlabel('Right ascension [degrees]', fontsize = 20)
ax.set_ylabel('Declination [degrees]', fontsize = 20)
plt.xlim([-30, 30])
plt.ylim([-30, 30])
plt.xticks(fontsize = 16)
plt.yticks(fontsize = 16)

plt.show()

## Focal plane (origin at optical axis)

In [ ]:
xFP = np.array([])
yFP = np.array([])

offset = 4

# Calculate the coordinates of the telescope pointings (for the 4 telescope groups) 
# in the focal-plane reference frame of group 1

for group in range(numGroups):

    xFP1, yFP1 = rf.skyToFocalPlaneCoordinates(radians(raTelescope[group] + raPlatform), radians(decTelescope[group] + decPlatform), radians(raPlatform), radians(decPlatform), 0,  \
                                              radians(tiltAngles[0]), radians(azimuthAngles[0]), 0, focalLength)
    xFP = np.append(xFP, xFP1)
    yFP = np.append(yFP, yFP1)
    
# Make sure the average of the telescope pointings of the 4 telescope groups is at the origin of the reference frame

xAvg = np.mean(xFP)
yAvg = np.mean(yFP)

xFP -= xAvg
yFP -= yAvg


# Rotate everything to make sure that the square connecting the telescope pointings of the 4 telescope groups
# are aligned with the axis of the reference frame.  The telescope pointing for telescope group 1 is in Q2, but
# this is an arbitrary choice.

# for group in range(numGroups):
#     
#     xRot, yRot = rotate_origin_only(xFP[group], yFP[group], -45)
#     xFP[group] = xRot
#     yFP[group] = yRot
    
meanDist = (np.mean(np.absolute(xFP)) + np.mean(np.absolute(yFP))) / 2.0

for group in range(numGroups):
    
    xFP[group] = sign(xFP[group]) * meanDist
    yFP[group] = sign(yFP[group]) * meanDist

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111)

plt.plot([0], [0], "ro")

plt.plot(xFP, yFP, "bo")

for index in range(numGroups):
    ax.text(xFP[index] + offset, yFP[index] + offset, "Group " + str(index + 1), fontsize = 20)

ax.text(0 + offset, 0 + offset, "$z_{PLM}$", fontsize = 20)


circles = []
for group in range(numGroups):
    circles.append(sg.Point(xFP[group], yFP[group]).buffer(fovMm))

for index in range(numGroups):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

ax.set_title('Telescope pointing w.r.t. platform pointing', fontsize = 24)
ax.set_xlabel('$x_{FP}$ [mm]', fontsize = 20)
ax.set_ylabel('$y_{FP}$ [mm]', fontsize = 20)
plt.xlim([-120, 120])
plt.ylim([-120, 120])
plt.xticks(fontsize = 16)
plt.yticks(fontsize = 16)

plt.show()

# Focal plane of individual telescope

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111)

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    ax.text(np.mean(cornersX) + offset, np.mean(cornersY) + offset, "CCD " + ccdCode, fontsize = 20)
    
    cornersX = np.append(cornersX, cornersX[0])
    cornersY = np.append(cornersY, cornersY[0])
    
    if ccdCode == "1":
        plt.plot(cornersX, cornersY, color="b", label="CCD footprint")
        ax.arrow(cornersX[0], cornersY[0], -30, 0, head_width=3, head_length=3, fc='k', ec='k', linewidth=2)
        ax.arrow(cornersX[0], cornersY[0], 0, -30, head_width=3, head_length=3, fc='k', ec='k', linewidth=2)
    else:
        plt.plot(cornersX, cornersY, color="b")
    if ccdCode == "2":
        ax.arrow(cornersX[0], cornersY[0], 30, 0, head_width=3, head_length=3, fc='k', ec='k', linewidth=2)
        ax.arrow(cornersX[0], cornersY[0], 0, -30, head_width=3, head_length=3, fc='k', ec='k', linewidth=2)
    elif ccdCode == "3":
        ax.arrow(cornersX[0], cornersY[0], 30, 0, head_width=3, head_length=3, fc='k', ec='k', linewidth=2)
        ax.arrow(cornersX[0], cornersY[0], 0, 30, head_width=3, head_length=3, fc='k', ec='k', linewidth=2)
    elif ccdCode == "4":
        ax.arrow(cornersX[0], cornersY[0], -30, 0, head_width=3, head_length=3, fc='k', ec='k', linewidth=2)
        ax.arrow(cornersX[0], cornersY[0], 0, 30, head_width=3, head_length=3, fc='k', ec='k', linewidth=2)
        
circ = plt.Circle((0, 0), radius=fovMm, color = "none", linewidth = 2, label="Telescope FOV")
ax.add_patch(circ)
circ.set_edgecolor("r")
circ.set_facecolor("lightgray")

plt.plot([0], [0], "rx", label="Optical axis")

plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
ax.set_title('CCDs in camera focal plane', fontsize = 24)
ax.set_xlabel('$x_{FP}$ [mm]', fontsize = 20)
ax.set_ylabel('$y_{FP} [mm]$', fontsize = 20)
plt.xticks(fontsize = 16)
plt.yticks(fontsize = 16)

plt.show()

# Telescope groups

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111)

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    cornersX = np.append(cornersX, cornersX[0])
    cornersY = np.append(cornersY, cornersY[0])
    
    for group in range(numGroups):
        
        if ccdCode == "1":
            plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o", label = "Group " + str(group + 1))
        else:
            plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o")
        plt.plot(cornersX + xFP[group], cornersY + yFP[group], color=colors[group])
        
plt.plot([0], [0], "kx", label="$z_{PLM}$")

plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
ax.set_title('CCDs of telescope groups', fontsize = 24)
ax.set_xlabel('$x_{FP}$ [mm]', fontsize = 20)
ax.set_ylabel('$y_{FP} [mm]$', fontsize = 20)
plt.xticks(fontsize = 16)
plt.yticks(fontsize = 16)

plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(xFP[group], yFP[group]).buffer(fovMm))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for group in range(numGroups):
    
    plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o")
    
    circ = plt.Circle((xFP[group], yFP[group]), radius = fovMm, color = "none", linewidth = 2, label = "Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", label="$z_{PLM}$")

plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))   
ax.set_title('FOV of telescope groups', fontsize = 24)
ax.set_xlabel('$x_{FP}$ [mm]', fontsize = 20)
ax.set_ylabel('$y_{FP} [mm]$', fontsize = 20)
plt.xticks(fontsize = 16)
plt.yticks(fontsize = 16)

plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111)

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    cornersX = np.append(cornersX, cornersX[0])
    cornersY = np.append(cornersY, cornersY[0])
    
    for group in range(numGroups):
        
        if ccdCode == "1":
            plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o")
        else:
            plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o")
        
        plt.plot(cornersX + xFP[group], cornersY + yFP[group], color=colors[group])
        
        offX = 75
        if (ccdCode == "3") or (ccdCode == "2"):
            offX = 70
        offY = 75
        if (ccdCode == "4") or (ccdCode == "1"):
            offY = 70
        ax.text(xFP[group] + sign(cornersX[3]) * offX, yFP[group] + sign(cornersY[3]) * offY, ccdCode, fontsize = 20, color=colors[group])
        
plt.plot([0], [0], "kx", mew=5, ms=10, label="$z_{PLM}$")


circles = []
for group in range(numGroups):
    circles.append(sg.Point(xFP[group], yFP[group]).buffer(fovMm))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for group in range(numGroups):
    
    plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o")
    
    circ = plt.Circle((xFP[group], yFP[group]), radius = fovMm, color = "none", linewidth = 2, label = "Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
ax.set_title('CCDs of telescope groups', fontsize = 24)
ax.set_xlabel('$x_{FP}$ [mm]', fontsize = 20)
ax.set_ylabel('$y_{FP} [mm]$', fontsize = 20)
plt.xticks(fontsize = 16)
plt.yticks(fontsize = 16)

plt.show()

In [ ]:
xPixels = np.copy(xFP)
yPixels = np.copy(yFP)

for group in range(numGroups):
    
    xPixels[group] = mm2pixels(xPixels[group])     # [mm] -> [pixels]
    yPixels[group] = mm2pixels(yPixels[group])     # [mm] -> [pixels]

# Helper Functions

In [ ]:
def highlightCcd1(refGroup):
    
    """
    Plot the four CCDs of the four telescope groups, highlighting CCD 1 of the given group.
    
    :param refGroup: Telescope group for which to highlight CCD 1.
    """
    
    ccdRefCode = "1"
    
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111)

    for ccdCode in ccdCodes:
    
        cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
        cornersX = np.append(cornersX, cornersX[0])
        cornersY = np.append(cornersY, cornersY[0])
    
        for group in range(numGroups):
        
            if ccdCode == ccdRefCode:
                plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o", label="Group " + str(group + 1))
            else:
                plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o")
        
            plt.plot(cornersX + xFP[group], cornersY + yFP[group], color=colors[group])
        
            offX = 70
            offY = 70
            
            ax.text(xFP[group] + sign(cornersX[3]) * offX, yFP[group] + sign(cornersY[3]) * offY, ccdCode, fontsize = 20, color=colors[group])
        
            if (group == refGroup - 1) and (ccdCode == ccdRefCode):
            
                plt.plot([cornersX[0] + xFP[group]], [cornersY[0] + yFP[group]], "kx", mew=5, ms=10, label="CCD origin")
            
                dim = 4510 * pixelSize / 1000.
                rect = plt.Rectangle((cornersX[2] + xFP[group], cornersY[2] + yFP[group]), dim, dim, alpha = 0.2)
                ax.add_patch(rect)
                rect.set_edgecolor(colors[group])
                rect.set_facecolor(colors[group])
            
                ax.arrow(cornersX[0] + xFP[group], cornersY[0] + yFP[group], -30, 0, head_width=5, head_length=5, fc='k', ec='k', linewidth=2)
                ax.arrow(cornersX[0] + xFP[group], cornersY[0] + yFP[group], 0, -30, head_width=5, head_length=5, fc='k', ec='k', linewidth=2)
    
    plt.plot([0], [0], "kx", label="$z_{PLM}$")

    plt.legend(prop={"size": 20}, bbox_to_anchor=(1.0, 1.0))
    title = "CCD " + refCcdCode + " in group " + str(refGroup)
    ax.set_title(title, fontsize = 24)
    ax.set_xlabel("$x_{FP}$ [mm]", fontsize = 20)
    ax.set_ylabel("$y_{FP} [mm]$", fontsize = 20)
    plt.xticks(fontsize = 16)
    plt.yticks(fontsize = 16)

    plt.show()

In [ ]:
def highlightCcd2(refGroup):
    
    """
    Plot the four CCDs of the four telescope groups, highlighting CCD 2 of the given group.
    
    :param refGroup: Telescope group for which to highlight CCD 2.
    """
    
    refCcdCode = "2"
    
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111)

    for ccdCode in ccdCodes:
    
        cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
        cornersX = np.append(cornersX, cornersX[0])
        cornersY = np.append(cornersY, cornersY[0])
    
        for group in range(numGroups):
        
            if ccdCode == refCcdCode:
                plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o", label="Group " + str(group + 1))
            else:
                plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o")
        
            plt.plot(cornersX + xFP[group], cornersY + yFP[group], color=colors[group])
        
            offX = 70
            offY = 70
            
            ax.text(xFP[group] + sign(cornersX[3]) * offX, yFP[group] + sign(cornersY[3]) * offY, ccdCode, fontsize = 20, color=colors[group])
        
            if (group == refGroup - 1) and (ccdCode == refCcdCode):
            
                plt.plot([cornersX[0] + xFP[group]], [cornersY[0] + yFP[group]], "kx", mew=5, ms=10, label="CCD origin")
            
                dim = 4510 * pixelSize / 1000.
                rect = plt.Rectangle((cornersX[1] + xFP[group], cornersY[1] + yFP[group]), dim, dim, alpha = 0.2)
                ax.add_patch(rect)
                rect.set_edgecolor(colors[group])
                rect.set_facecolor(colors[group])
            
                ax.arrow(cornersX[0] + xFP[group], cornersY[0] + yFP[group], 30, 0, head_width=5, head_length=5, fc='k', ec='k', linewidth=2)
                ax.arrow(cornersX[0] + xFP[group], cornersY[0] + yFP[group], 0, -30, head_width=5, head_length=5, fc='k', ec='k', linewidth=2)
    
    plt.plot([0], [0], "kx", label="$z_{PLM}$")

    plt.legend(prop={"size": 20}, bbox_to_anchor=(1.0, 1.0))
    title = "CCD " + refCcdCode + " in group " + str(refGroup)
    ax.set_title(title, fontsize = 24)
    ax.set_xlabel("$x_{FP}$ [mm]", fontsize = 20)
    ax.set_ylabel("$y_{FP} [mm]$", fontsize = 20)
    plt.xticks(fontsize = 16)
    plt.yticks(fontsize = 16)

    plt.show()

In [ ]:
def highlightCcd3(refGroup):
    
    """
    Plot the four CCDs of the four telescope groups, highlighting CCD 3 of the given group.
    
    :param refGroup: Telescope group for which to highlight CCD 3.
    """
    
    refCcdCode = "3"
    
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111)

    for ccdCode in ccdCodes:
    
        cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
        cornersX = np.append(cornersX, cornersX[0])
        cornersY = np.append(cornersY, cornersY[0])
    
        for group in range(numGroups):
        
            if ccdCode == refCcdCode:
                plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o", label="Group " + str(group + 1))
            else:
                plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o")
        
            plt.plot(cornersX + xFP[group], cornersY + yFP[group], color=colors[group])
        
            offX = 70
            offY = 70
            
            ax.text(xFP[group] + sign(cornersX[3]) * offX, yFP[group] + sign(cornersY[3]) * offY, ccdCode, fontsize = 20, color=colors[group])
        
            if (group == refGroup - 1) and (ccdCode == refCcdCode):
            
                plt.plot([cornersX[0] + xFP[group]], [cornersY[0] + yFP[group]], "kx", mew=5, ms=10, label="CCD origin")
            
                dim = 4510 * pixelSize / 1000.
                rect = plt.Rectangle((cornersX[0] + xFP[group], cornersY[0] + yFP[group]), dim, dim, alpha = 0.2)
                ax.add_patch(rect)
                rect.set_edgecolor(colors[group])
                rect.set_facecolor(colors[group])
            
                ax.arrow(cornersX[0] + xFP[group], cornersY[0] + yFP[group], 30, 0, head_width=5, head_length=5, fc='k', ec='k', linewidth=2)
                ax.arrow(cornersX[0] + xFP[group], cornersY[0] + yFP[group], 0, 30, head_width=5, head_length=5, fc='k', ec='k', linewidth=2)
    
    plt.plot([0], [0], "kx", label="$z_{PLM}$")

    plt.legend(prop={"size": 20}, bbox_to_anchor=(1.0, 1.0))
    title = "CCD " + refCcdCode + " in group " + str(refGroup)
    ax.set_title(title, fontsize = 24)
    ax.set_xlabel("$x_{FP}$ [mm]", fontsize = 20)
    ax.set_ylabel("$y_{FP} [mm]$", fontsize = 20)
    plt.xticks(fontsize = 16)
    plt.yticks(fontsize = 16)

    plt.show()

In [ ]:
def highlightCcd4(refGroup):
    
    """
    Plot the four CCDs of the four telescope groups, highlighting CCD 4 of the given group.
    
    :param refGroup: Telescope group for which to highlight CCD 4.
    """
    
    refCcdCode = "4"
    
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111)

    for ccdCode in ccdCodes:
    
        cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
        cornersX = np.append(cornersX, cornersX[0])
        cornersY = np.append(cornersY, cornersY[0])
    
        for group in range(numGroups):
        
            if ccdCode == refCcdCode:
                plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o", label="Group " + str(group + 1))
            else:
                plt.plot([xFP[group]], [yFP[group]], color=colors[group], marker="o")
        
            plt.plot(cornersX + xFP[group], cornersY + yFP[group], color=colors[group])
        
            offX = 70
            offY = 70
            
            ax.text(xFP[group] + sign(cornersX[3]) * offX, yFP[group] + sign(cornersY[3]) * offY, ccdCode, fontsize = 20, color=colors[group])
        
            if (group == refGroup - 1) and (ccdCode == refCcdCode):
            
                plt.plot([cornersX[0] + xFP[group]], [cornersY[0] + yFP[group]], "kx", mew=5, ms=10, label="CCD origin")
            
                dim = 4510 * pixelSize / 1000.
                rect = plt.Rectangle((cornersX[3] + xFP[group], cornersY[3] + yFP[group]), dim, dim, alpha = 0.2)
                ax.add_patch(rect)
                rect.set_edgecolor(colors[group])
                rect.set_facecolor(colors[group])
            
                ax.arrow(cornersX[0] + xFP[group], cornersY[0] + yFP[group], -30, 0, head_width=5, head_length=5, fc='k', ec='k', linewidth=2)
                ax.arrow(cornersX[0] + xFP[group], cornersY[0] + yFP[group], 0, 30, head_width=5, head_length=5, fc='k', ec='k', linewidth=2)
    
    plt.plot([0], [0], "kx", label="$z_{PLM}$")

    plt.legend(prop={"size": 20}, bbox_to_anchor=(1.0, 1.0))
    title = "CCD " + refCcdCode + " in group " + str(refGroup)
    ax.set_title(title, fontsize = 24)
    ax.set_xlabel("$x_{FP}$ [mm]", fontsize = 20)
    ax.set_ylabel("$y_{FP} [mm]$", fontsize = 20)
    plt.xticks(fontsize = 16)
    plt.yticks(fontsize = 16)

    plt.show()

In [ ]:
def highlightCcd(refCcdCode, refGroup):
    
    """
    Plot the four CCDs of the four telescope groups, highlighting the given CCD of the given group.
    
    :param refCcdCode: CCD to highlight for the given telescope group.
    
    :param refGroup: Telescope group for which to highlight the given CCD.
    """
    
    if refCcdCode == "1":
        highlightCcd1(refGroup)
    
    elif refCcdCode == "2":
        highlightCcd2(refGroup)
        
    elif refCcdCode == "3":
        highlightCcd3(refGroup)
        
    elif refCcdCode == "4":
        highlightCcd4(refGroup)

# Group 1

In [ ]:
refGroup = 1

## CCD 1

In [ ]:
refCcdCode = "1"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(-(xPixels[group] - offsetX), -(yPixels[group] - offsetY)).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]
    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(-arrayX, -arrayY, color=colors[group])
        
        off = 4000
        ax.text(-(xPixels[group] - offsetX + sign(cornersX[3]) * off), -(yPixels[group] - offsetY + sign(cornersY[3]) * off), ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([-(xPixels[group] - offsetX)], [-(yPixels[group] - offsetY)], color=colors[group], marker="o")
    
    circ = plt.Circle(((-(xPixels[group] - offsetX)), (-(yPixels[group] - offsetY))), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={"size": 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel("column [pixels]", fontsize = 24)
ax.set_ylabel("row [pixel]", fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-5000, 9000, 1000))
ax.set_yticks(np.arange(0, 13000, 1000))
ax.arrow(0, 0, 7750, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 12250, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

## CCD 2

In [ ]:
refCcdCode = "2"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(-(yPixels[group] - offsetY), xPixels[group] - offsetX).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]
    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(-arrayY, arrayX, color=colors[group])
        
        off = 4000
        ax.text(-(yPixels[group] - offsetY + sign(cornersY[3]) * off), xPixels[group] - offsetX + sign(cornersX[3]) * off, ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([-(yPixels[group] - offsetY)], [xPixels[group] - offsetX], color=colors[group], marker="o")
    
    circ = plt.Circle(((-(yPixels[group] - offsetY)), (xPixels[group] - offsetX)), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel('column [pixels]', fontsize = 24)
ax.set_ylabel('row [pixel]', fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-5000, 9000, 1000))
ax.set_yticks(np.arange(-4000, 10000, 1000))
ax.arrow(0, 0, 7750, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 9000, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

## CCD 3

In [ ]:
refCcdCode = "3"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(xPixels[group] - offsetX, yPixels[group] - offsetY).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]

    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(arrayX, arrayY, color=colors[group])
        
        off = 4000
        ax.text(xPixels[group] - offsetX + sign(cornersX[3]) * off, yPixels[group] - offsetY + sign(cornersY[3]) * off, ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([xPixels[group] - offsetX], [yPixels[group] - offsetY], color=colors[group], marker="o")
    
    circ = plt.Circle(((xPixels[group] - offsetX), (yPixels[group] - offsetY)), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={"size": 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel("column [pixels]", fontsize = 24)
ax.set_ylabel("row [pixel]", fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-8000, 6000, 1000))
ax.set_yticks(np.arange(-4000, 10000, 1000))

ax.arrow(0, 0, 4500, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 9250, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

## CCD 4

In [ ]:
refCcdCode = "4"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(yPixels[group] - offsetY, -(xPixels[group] - offsetX)).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]
    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(arrayY, -arrayX, color=colors[group])
        
        off = 4000
        ax.text(yPixels[group] - offsetY + sign(cornersY[3]) * off, -(xPixels[group] - offsetX + sign(cornersX[3]) * off), ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([yPixels[group] - offsetY], [-(xPixels[group] - offsetX)], color=colors[group], marker="o")
    
    circ = plt.Circle(((yPixels[group] - offsetY), (-(xPixels[group] - offsetX))), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel('column [pixels]', fontsize = 24)
ax.set_ylabel('row [pixel]', fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-8000, 6000, 1000))
ax.set_yticks(np.arange(0, 13000, 1000))

ax.arrow(0, 0, 4500, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 12250, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

# Group 2

In [ ]:
refGroup = 2

## CCD 1

In [ ]:
refCcdCode = "1"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(-(xPixels[group] - offsetX), -(yPixels[group] - offsetY)).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]
    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(-arrayX, -arrayY, color=colors[group])
        
        off = 4000
        ax.text(-(xPixels[group] - offsetX + sign(cornersX[3]) * off), -(yPixels[group] - offsetY + sign(cornersY[3]) * off), ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([-(xPixels[group] - offsetX)], [-(yPixels[group] - offsetY)], color=colors[group], marker="o")
    
    circ = plt.Circle(((-(xPixels[group] - offsetX)), (-(yPixels[group] - offsetY))), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel('column [pixels]', fontsize = 24)
ax.set_ylabel('row [pixel]', fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-8000, 6000, 1000))
ax.set_yticks(np.arange(0, 13000, 1000))
ax.arrow(0, 0, 4500, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 12500, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

## CCD 2

In [ ]:
refCcdCode = "2"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(-(yPixels[group] - offsetY), xPixels[group] - offsetX).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]
    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(-arrayY, arrayX, color=colors[group])
        
        off = 4000
        ax.text(-(yPixels[group] - offsetY + sign(cornersY[3]) * off), xPixels[group] - offsetX + sign(cornersX[3]) * off, ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([-(yPixels[group] - offsetY)], [xPixels[group] - offsetX], color=colors[group], marker="o")
    
    circ = plt.Circle(((-(yPixels[group] - offsetY)), (xPixels[group] - offsetX)), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel('column [pixels]', fontsize = 24)
ax.set_ylabel('row [pixel]', fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-5000, 9000, 1000))
ax.set_yticks(np.arange(0, 13000, 1000))
ax.arrow(0, 0, 7750, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 12250, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

## CCD 3

In [ ]:
refCcdCode = "3"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(xPixels[group] - offsetX, yPixels[group] - offsetY).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]

    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(arrayX, arrayY, color=colors[group])
        
        off = 4000
        ax.text(xPixels[group] - offsetX + sign(cornersX[3]) * off, yPixels[group] - offsetY + sign(cornersY[3]) * off, ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([xPixels[group] - offsetX], [yPixels[group] - offsetY], color=colors[group], marker="o")
    
    circ = plt.Circle(((xPixels[group] - offsetX), (yPixels[group] - offsetY)), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={"size": 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel("column [pixels]", fontsize = 24)
ax.set_ylabel("row [pixel]", fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-5000, 9000, 1000))
ax.set_yticks(np.arange(-4000, 10000, 1000))

ax.arrow(0, 0, 8000, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 9250, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

## CCD 4

In [ ]:
refCcdCode = "4"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(yPixels[group] - offsetY, -(xPixels[group] - offsetX)).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]
    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(arrayY, -arrayX, color=colors[group])
        
        off = 4000
        ax.text(yPixels[group] - offsetY + sign(cornersY[3]) * off, -(xPixels[group] - offsetX + sign(cornersX[3]) * off), ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([yPixels[group] - offsetY], [-(xPixels[group] - offsetX)], color=colors[group], marker="o")
    
    circ = plt.Circle(((yPixels[group] - offsetY), (-(xPixels[group] - offsetX))), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel('column [pixels]', fontsize = 24)
ax.set_ylabel('row [pixel]', fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-8000, 6000, 1000))
ax.set_yticks(np.arange(-4000, 10000, 1000))
ax.arrow(0, 0, 4500, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 9250, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

# Group 3

In [ ]:
refGroup = 3

## CCD 1

In [ ]:
refCcdCode = "1"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(-(xPixels[group] - offsetX), -(yPixels[group] - offsetY)).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]
    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(-arrayX, -arrayY, color=colors[group])
        
        off = 4000
        ax.text(-(xPixels[group] - offsetX + sign(cornersX[3]) * off), -(yPixels[group] - offsetY + sign(cornersY[3]) * off), ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([-(xPixels[group] - offsetX)], [-(yPixels[group] - offsetY)], color=colors[group], marker="o")
    
    circ = plt.Circle(((-(xPixels[group] - offsetX)), (-(yPixels[group] - offsetY))), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel('column [pixels]', fontsize = 24)
ax.set_ylabel('row [pixel]', fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-8000, 6000, 1000))
ax.set_yticks(np.arange(-4000, 10000, 1000))
ax.arrow(0, 0, 4500, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 9250, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

## CCD 2

In [ ]:
refCcdCode = "2"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(-(yPixels[group] - offsetY), xPixels[group] - offsetX).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]
    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(-arrayY, arrayX, color=colors[group])
        
        off = 4000
        ax.text(-(yPixels[group] - offsetY + sign(cornersY[3]) * off), xPixels[group] - offsetX + sign(cornersX[3]) * off, ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([-(yPixels[group] - offsetY)], [xPixels[group] - offsetX], color=colors[group], marker="o")
    
    circ = plt.Circle(((-(yPixels[group] - offsetY)), (xPixels[group] - offsetX)), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel('column [pixels]', fontsize = 24)
ax.set_ylabel('row [pixel]', fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-8000, 6000, 1000))
ax.set_yticks(np.arange(0, 13000, 1000))
ax.arrow(0, 0, 4500, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 12250, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

## CCD 3

In [ ]:
refCcdCode = "3"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(xPixels[group] - offsetX, yPixels[group] - offsetY).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]

    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(arrayX, arrayY, color=colors[group])
        
        off = 4000
        ax.text(xPixels[group] - offsetX + sign(cornersX[3]) * off, yPixels[group] - offsetY + sign(cornersY[3]) * off, ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([xPixels[group] - offsetX], [yPixels[group] - offsetY], color=colors[group], marker="o")
    
    circ = plt.Circle(((xPixels[group] - offsetX), (yPixels[group] - offsetY)), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={"size": 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel("column [pixels]", fontsize = 24)
ax.set_ylabel("row [pixel]", fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-5000, 9000, 1000))
ax.set_yticks(np.arange(0, 13000, 1000))

ax.arrow(0, 0, 8000, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 12500, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

## CCD 4

In [ ]:
refCcdCode = "4"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(yPixels[group] - offsetY, -(xPixels[group] - offsetX)).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]
    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(arrayY, -arrayX, color=colors[group])
        
        off = 4000
        ax.text(yPixels[group] - offsetY + sign(cornersY[3]) * off, -(xPixels[group] - offsetX + sign(cornersX[3]) * off), ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([yPixels[group] - offsetY], [-(xPixels[group] - offsetX)], color=colors[group], marker="o")
    
    circ = plt.Circle(((yPixels[group] - offsetY), (-(xPixels[group] - offsetX))), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel('column [pixels]', fontsize = 24)
ax.set_ylabel('row [pixel]', fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-5000, 9000, 1000))
ax.set_yticks(np.arange(-4000, 10000, 1000))
ax.arrow(0, 0, 7750, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 9250, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

# Group 4

In [ ]:
refGroup = 4

## CCD 1

In [ ]:
refCcdCode = "1"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(-(xPixels[group] - offsetX), -(yPixels[group] - offsetY)).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]
    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(-arrayX, -arrayY, color=colors[group])
        
        off = 4000
        ax.text(-(xPixels[group] - offsetX + sign(cornersX[3]) * off), -(yPixels[group] - offsetY + sign(cornersY[3]) * off), ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([-(xPixels[group] - offsetX)], [-(yPixels[group] - offsetY)], color=colors[group], marker="o")
    
    circ = plt.Circle(((-(xPixels[group] - offsetX)), (-(yPixels[group] - offsetY))), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel('column [pixels]', fontsize = 24)
ax.set_ylabel('row [pixel]', fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-5000, 9000, 1000))
ax.set_yticks(np.arange(-4000, 10000, 1000))
ax.arrow(0, 0, 8000, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 9250, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

## CCD 2

In [ ]:
refCcdCode = "2"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(-(yPixels[group] - offsetY), xPixels[group] - offsetX).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]
    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(-arrayY, arrayX, color=colors[group])
        
        off = 4000
        ax.text(-(yPixels[group] - offsetY + sign(cornersY[3]) * off), xPixels[group] - offsetX + sign(cornersX[3]) * off, ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([-(yPixels[group] - offsetY)], [xPixels[group] - offsetX], color=colors[group], marker="o")
    
    circ = plt.Circle(((-(yPixels[group] - offsetY)), (xPixels[group] - offsetX)), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel('column [pixels]', fontsize = 24)
ax.set_ylabel('row [pixel]', fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-8000, 6000, 1000))
ax.set_yticks(np.arange(-4000, 10000, 1000))
ax.arrow(0, 0, 4500, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 9250, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

## CCD 3

In [ ]:
refCcdCode = "3"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(xPixels[group] - offsetX, yPixels[group] - offsetY).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]

    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(arrayX, arrayY, color=colors[group])
        
        off = 4000
        ax.text(xPixels[group] - offsetX + sign(cornersX[3]) * off, yPixels[group] - offsetY + sign(cornersY[3]) * off, ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([xPixels[group] - offsetX], [yPixels[group] - offsetY], color=colors[group], marker="o")
    
    circ = plt.Circle(((xPixels[group] - offsetX), (yPixels[group] - offsetY)), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={"size": 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel("column [pixels]", fontsize = 24)
ax.set_ylabel("row [pixel]", fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-8000, 6000, 1000))
ax.set_yticks(np.arange(0, 13000, 1000))

ax.arrow(0, 0, 4500, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 12500, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()

## CCD 4

In [ ]:
refCcdCode = "4"

In [ ]:
highlightCcd(refCcdCode, refGroup)

In [ ]:
index = 0

cornersX, cornersY = rf.computeCCDcornersInFocalPlane(refCcdCode, pixelSize)
offsetX = mm2pixels(cornersX[index]) + xPixels[refGroup - 1]
offsetY = mm2pixels(cornersY[index]) + yPixels[refGroup - 1]

fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

circles = []
for group in range(numGroups):
    circles.append(sg.Point(yPixels[group] - offsetY, -(xPixels[group] - offsetX)).buffer(fovPixels))

for index in range(numCorners):
    
    one = circles[index].intersection(circles[index])
    ax.add_patch(descartes.PolygonPatch(one, fc='gray', ec='none', alpha=0.2))
    
    two = circles[index].intersection(circles[(index + 1) % numCorners])
    ax.add_patch(descartes.PolygonPatch(two, fc='gray', ec='none', alpha=0.2))
    
    three = circles[index].intersection(circles[(index + 1) % numCorners]).intersection(circles[(index + 2) % numCorners])
    ax.add_patch(descartes.PolygonPatch(three, fc='gray', ec='none', alpha=0.2))
    
four = circles[0].intersection(circles[1]).intersection(circles[2]).intersection(circles[3])
ax.add_patch(descartes.PolygonPatch(four, fc='gray', ec='none', alpha=0.2))

for ccdCode in ccdCodes:
    
    cornersX, cornersY = rf.computeCCDcornersInFocalPlane(ccdCode, pixelSize)
    
    for corner in range(numCorners):
        cornersX[corner] = mm2pixels(cornersX[corner])
        cornersY[corner] = mm2pixels(cornersY[corner])
    
    cornersX = np.append(cornersX, cornersX[0])     # [mm]
    cornersY = np.append(cornersY, cornersY[0])     # [mm]
    
    for group in range(numGroups):
        
        arrayX = cornersX + xPixels[group] - offsetX
        arrayY = cornersY + yPixels[group] - offsetY
        
        for index in range(5):
            arrayX[index] = (arrayX[index])
            arrayY[index] = (arrayY[index])
        
        plt.plot(arrayY, -arrayX, color=colors[group])
        
        off = 4000
        ax.text(yPixels[group] - offsetY + sign(cornersY[3]) * off, -(xPixels[group] - offsetX + sign(cornersX[3]) * off), ccdCode, fontsize = 24, color=colors[group])
        
for group in range(numGroups):
    
    plt.plot([yPixels[group] - offsetY], [-(xPixels[group] - offsetX)], color=colors[group], marker="o")
    
    circ = plt.Circle(((yPixels[group] - offsetY), (-(xPixels[group] - offsetX))), radius = fovPixels, color = "none", linewidth = 2, label="Group " + str(group + 1))
    ax.add_patch(circ)
    circ.set_edgecolor(colors[group])
    circ.set_facecolor("none")

plt.plot([0], [0], "kx", mew=5, ms=10, label="CCD origin")
            
plt.legend(prop={'size': 20}, bbox_to_anchor=(1.0, 1.0))
title = "CCD " + refCcdCode + " in group " + str(refGroup)
ax.set_title(title, fontsize = 28)
ax.set_xlabel('column [pixels]', fontsize = 24)
ax.set_ylabel('row [pixel]', fontsize = 24)
plt.xticks(fontsize = 20)
plt.yticks(fontsize = 20)

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

ax.tick_params(which='both', width=2)
ax.tick_params(which='major', length=7)
ax.tick_params(which='minor', length=4, color='r')
ax.grid(True)
ax.set_xticks(np.arange(-5000, 9000, 1000))
ax.set_yticks(np.arange(0, 13000, 1000))
ax.arrow(0, 0, 7750, 0, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)
ax.arrow(0, 0, 0, 12250, head_width=200, head_length=200, fc='k', ec='k', linewidth=4)

plt.show()